In [1]:
CellSIUS = function(mat.norm,group_id,min_n_cells=10, min_fc = 2,
                         corr_cutoff = NULL, iter=0, max_perc_cells = 50,
                         fc_between_cutoff = 1,mcl_path = "~/local/bin/mcl"){
  options(warn=-1)
  if(is.matrix(mat.norm)!=TRUE) stop("mat.norm is not of 'matrix' class")
  if(length(unique(group_id))<2) stop("The number of clusters in 'group_id' must be > 1")
  if(!file.exists(mcl_path)) stop("I don't find mcl executable. External UNIX installation of MCL is required: https://micans.org/mcl/")
  if(is.null(rownames(mat.norm))| is.null(colnames(mat.norm))) stop("Column and row names of mat.norm matrix must be non-emptys")
  group_id = data.table::data.table(cell_idx = colnames(mat.norm),main_cluster=as.character(group_id))
  expr_dt = data.table::data.table(gene_id = rownames(mat.norm),mat.norm)
  expr_dt_melt = data.table::melt(expr_dt,id.vars="gene_id",val="expr",var="cell_idx")
  expr_dt_melt = merge(expr_dt_melt,group_id, by="cell_idx")

  #Identify genes with significant bimodal distribution

  expr_dt_melt[,c("N_cells","within_p","pos0","pos1","Dpos"):=cellsius_find_bimodal_genes(expr,min_n_cells = min_n_cells, max_perc_cells = max_perc_cells),by=c('gene_id','main_cluster')]
  expr_dt_melt[,sig := within_p<100 & Dpos > min_fc]
  expr_dt_melt[sig==T, within_adj_p:=p.adjust(within_p),by=c('cell_idx')] #correct for multiple testing, only consider genes where test has actually been run
  expr_dt_melt[,sig:=within_adj_p<0.1]
  expr_dt_melt = expr_dt_melt[gene_id %in% expr_dt_melt[!is.na(sig) & sig==T]$gene_id]

  # If no bimodal gene were found, exit and return NA
  if(dim(expr_dt_melt)[1] == 0){
    print("No genes with bimodal distribution found, returning NA.")
    return(NA)
  }
  # Check whether these genes are specific to the subcluster

  for(clust in unique(expr_dt_melt$main_cluster)){
    expr_dt_melt = expr_dt_melt[,paste0(clust,"_",c("p_between","fc")):=cellsius_test_cluster_specificity(expr,main_cluster,clust, fc_between_cutoff = fc_between_cutoff),by="gene_id"]
    expr_dt_melt[main_cluster==clust,keep:=(expr_dt_melt[main_cluster==clust][[paste0(clust,"_p_between")]] < 0.1)]
  }

  expr_dt_melt = expr_dt_melt[keep==TRUE & !is.na(sig)]

  # If there are still non-specific genes, discard them (this can happen for
  # very high expressed genes like mitochondrial genes)
  expr_dt_melt[,n_clust_per_gene:=length(unique(main_cluster)),by='gene_id']
  expr_dt_melt = expr_dt_melt[n_clust_per_gene==1]
  expr_dt_melt[,n_clust_per_gene:=NULL]

  # Identify correlated gene sets with MCL
  expr_dt_melt = expr_dt_melt[,gene_cluster:=0]
  expr_dt_melt = cellsius_find_gene_sets(expr_dt_melt, corr_cutoff = corr_cutoff,mcl_path=mcl_path)

  # discard gene sets that only contain one gene (those are assigned to cluster 0)
  expr_dt_melt = expr_dt_melt[gene_cluster !=0 ]

  if(dim(expr_dt_melt)[1] == 0){
    print("No subclusters found, returning NA.")
    return(NA)
  }

  # Extract cell subclusters
  expr_dt_melt[,sub_cluster:=main_cluster]
  expr_dt_melt[,mean_expr := mean(expr), by = c('main_cluster','gene_cluster','cell_idx')]
  expr_dt_melt[,sub_cluster:=cellsius_sub_cluster(mean_expr,sub_cluster,gene_cluster, iter=iter),by=c('main_cluster','gene_cluster')]

  # Check how many cells belong to the subgroup relative to the total cluster size.
  # If a sub cluster contains more than max_perc_cells cells, discard it.
  clust_list = expr_dt_melt[,list(sub = length(unique(cell_idx))) ,by=c('sub_cluster','main_cluster')]
  clust_list[,tot := sum(sub)/(length(sub_cluster)/2), by= 'main_cluster']
  clust_list = clust_list[grep('_1$',sub_cluster)]
  clust_list[,perc:=sub/tot*100]
  discard_sub_clust = clust_list[perc > max_perc_cells]$sub_cluster
  discard_sub_clust = append(discard_sub_clust,gsub('_1$','_0',discard_sub_clust))

  expr_dt_melt = expr_dt_melt[!sub_cluster%in%discard_sub_clust]

  keep.columns = c("cell_idx", "gene_id", "expr", "main_cluster", "N_cells", "Dpos", "sub_cluster","gene_cluster")
  expr_dt_melt = expr_dt_melt[,..keep.columns]
  setnames(expr_dt_melt, 'Dpos', 'log2FC')
  return(expr_dt_melt)
}

#' Identify genes with bimodal distribution
#' @description
#' Accessory function of \code{\link[CellSIUS]{CellSIUS}}
#' @keywords internal

##################################################
# STEP 1: Identify genes with bimodal distribution
##################################################
cellsius_find_bimodal_genes = function(expr, min_n_cells, max_perc_cells){

  #skip genes with 0 expression
  if(sum(expr)==0){
    return(list(-1,100,-1,-1,-1))
  }
  # run k-means
  k1d = Ckmeans.1d.dp::Ckmeans.1d.dp(expr,k=2)
  # check if a cluster with more than n cells exists
  indx = which(k1d$size>min_n_cells)
  if(length(indx)>1 ){

    # do statistic only if in pos2 cells are less than max_perc_cells% of the total cells in the cluster
    if(k1d$size[2] < round(length(expr)*max_perc_cells/100)){

      t1=tryCatch({t.test(expr[which(k1d$cluster==2)],y=expr[which(k1d$cluster==1)])},
                  error = function(cond){return(0)},
                  finally={}
      )

      if(!is.numeric(t1)){

        p1=t1$p.value
        N0=k1d$size[1] # number of cells where the gene is downregulated
        N1=k1d$size[2] # number of cells  where the gene is upregulated
        pos0=k1d$centers[1]
        pos1=k1d$centers[2]
        Dpos=pos1-pos0
        return(list(N1,p1,pos0,pos1,Dpos))
      } #else {print(paste("ttest failed, dpos = ",pos1-pos0))} # for testing
    }
  }
  # if no cluster was found, return a list of dummy values
  return(list(-1,100,-1,-1,-1))
}

#' Check whether the subset of genes are specific to one cell subgroup
#' @description
#' Accessory function of \code{\link[CellSIUS]{CellSIUS}}
#' @keywords internal


#############################################################################
# STEP 2: Check whether these genes are specific to one cell subgroup
############################################################################
cellsius_test_cluster_specificity = function(exprs, cluster, current_cluster, fc_between_cutoff){

  in_clust = which(cluster == current_cluster)
  k1d = Ckmeans.1d.dp::Ckmeans.1d.dp(exprs[in_clust],k=2)
  in_subclust = in_clust[which(k1d$cluster==2)]

  mean_in = mean(exprs[in_subclust])
  mean_out = mean(exprs[-in_subclust])
  mean_out_nozero = mean(exprs[-in_subclust][exprs[-in_subclust]>0])

  # If there are subclusters, but all cells outside the subcluster express 0,
  # set mean_out_nozero to 0
  if(length(in_subclust>0) && !any(exprs[-in_subclust]>0)){mean_out_nozero=0}

  fc = mean_in - mean_out

  ts = tryCatch({t.test(exprs[in_subclust],exprs[-in_clust])},
                error = function(cond){ return(0)})

  if(!is.numeric(ts)){pv = ts$p.value} else {
    #print(paste("ttest failed, fc = ",mean_in-mean_out_nozero)) #for testing only
    pv=999}

  if(!is.nan(mean_out_nozero) && (mean_in-mean_out_nozero < fc_between_cutoff)) pv = 999
  return(list(pv,fc))
}

#' MCL clustering to find correlated gene sets
#' @description
#' Accessory function of \code{\link[CellSIUS]{CellSIUS}}
#' @keywords internal

#####################################################
# STEP 3: MCL clustering to find correlated gene sets
#####################################################
cellsius_find_gene_sets = function(expr_dt_melt, corr_cutoff = NULL, min_corr = 0.35, max_corr = 0.5,mcl_path = mcl_path){
  for(clust in unique(expr_dt_melt$main_cluster)){

    if(length(unique(expr_dt_melt[main_cluster == clust]$gene_id))==1) { next }

    mat = data.table::dcast.data.table(expr_dt_melt[main_cluster==clust], gene_id ~ cell_idx,
                                       value.var = 'expr')
    mat = mat[rowSums(mat[,-1,with=F])!=0,]
    corr.mat = cor(t(mat[,-1,with=F]))
    dimnames(corr.mat) = list(mat$gene_id,mat$gene_id)

    if(is.null(corr_cutoff)){
      corr_cutoff = max(quantile(corr.mat[corr.mat!=1],0.95),min_corr)
      corr_cutoff = min(corr_cutoff, max_corr)}
    adj.corr = corr.mat
    adj.corr[adj.corr<corr_cutoff] = 0
    adj.corr[adj.corr>=corr_cutoff] = 1
    diag(adj.corr) = 0 # no self-loop for MCL

    graphs = igraph::get.data.frame( igraph::graph_from_adjacency_matrix(adj.corr), what = "edges") # gene connection for graphs

    # if a graph has no edges (i.e. all genes are uncorrelated),
    # assign all genes to cluster "0" and go to next iteration
    if(dim(graphs)[1]==0){
      expr_dt_melt = expr_dt_melt[main_cluster == clust, gene_cluster := 0]
      next
    }

    graphs = data.frame(graphs,CORR=sapply(seq(dim(graphs)[1]), function(i) corr.mat[graphs$from[i],graphs$to[i]] -corr_cutoff))
    write.table(graphs, file = "tmp.mcl.inp",row.names=F,col.names=F,sep = " ")
    system(paste0(mcl_path, " tmp.mcl.inp --abc -o tmp.mcl.out"))
    x = scan("tmp.mcl.out", what="", sep="\n")
    y = strsplit(x, "[[:space:]]+")
    y = lapply(seq(length(y)), function(i){
      tmp = sapply(seq(length(y[[i]])),function(j){
        gsub('\"','',y[[i]][j])
      })
    })

    for(i in seq(length(y))){
      if(length(y[[i]] > 1)){
        expr_dt_melt = expr_dt_melt[main_cluster==clust & gene_id %in% y[[i]],gene_cluster:=i]
      }
    }
  }

  return(expr_dt_melt)
}

#' Assign cells to subgroups
#' @description
#' Accessory function of \code{\link[CellSIUS]{CellSIUS}}
#' @keywords internal

############################################
# Step 4: Assign cells to subgroups
############################################
cellsius_sub_cluster = function(mean_expr,sub_cluster,gene_cluster, iter = iter){

  k1d = Ckmeans.1d.dp::Ckmeans.1d.dp(mean_expr,k=2)$cluster
  cells_sub = (k1d==2)

  if(iter == 0){return(paste0(sub_cluster,"_",gene_cluster,"_",as.numeric(cells_sub)))}

  # if iter is set higher than 0, a second step of kmeans clustering.
  # This will remove the lowest peak and can sometimes help to get a more
  # accurate classification.

  k1d = Ckmeans.1d.dp::Ckmeans.1d.dp(mean_expr[cells_sub],k=2)$cluster

  if (max(k1d)>1) {
    cells_sub[cells_sub] = (k1d==2)
    return(paste0(sub_cluster,"_",gene_cluster,"_",as.numeric(cells_sub)))
  }
  return(paste0(sub_cluster,"_",gene_cluster,"_",0))
}

In [2]:
CellSIUS_GetResults = function(CellSIUS.out){
   type = "summary"
  out = switch(type,
               "summary" = cellsius_print_summary(CellSIUS.out),
               "cells_per_clust" = cellsius_get_cells_per_subclust(CellSIUS.out))
}

#' Print Summary
#' @description
#' Accessory function of \code{\link[CellSIUS]{CellSIUS_GetResults}}
#' @keywords internal



cellsius_print_summary = function(CellSIUS.out){

  summary_list = list()

  cat('--------------------------------------------------------\n',
      'Summary of CellSIUS output\n',
      '--------------------------------------------------------\n\n')
  for(clust in unique(CellSIUS.out$main_cluster)){

    if(!any(CellSIUS.out[main_cluster==clust]$N_cells!=0)){
      next
    }

    cat('Main cluster: ', clust,  '\n', '---------------\n')
    subclusts = unique(CellSIUS.out[main_cluster==clust & gene_cluster!=0][order(gene_cluster)]$gene_cluster)

    for(subclust in subclusts){

      cells = unique(CellSIUS.out[main_cluster==clust & sub_cluster == paste(clust,subclust,1,sep="_")]$cell_idx)
      genes = unique(CellSIUS.out[main_cluster==clust & gene_cluster == subclust][,c("gene_id","log2FC")])
      n_cells = length(cells)
      subclust_list = list(cells = cells, genes = genes)

      cat('Subcluster: ', subclust, '\n',
          'Number of cells: ', n_cells,
          '\n Marker genes: \n')

      print(genes)
      cat('\n\n')

      subclust_name = paste(clust, subclust, sep='_')
      summary_list[[subclust_name]] = subclust_list

    }
  }
  return(summary_list)
}

#' Get cells per subcluster
#' @description
#' Accessory function of \code{\link[CellSIUS]{CellSIUS_GetResults}}
#' @keywords internal

cellsius_get_cells_per_subclust = function(CellSIUS.out){
  print('Number of cells per subcluster:')
  print(CellSIUS.out[,length(unique(cell_idx)),by="sub_cluster"])
  out = unique(CellSIUS.out[,length(unique(cell_idx)),by="sub_cluster"])
}

In [3]:
CellSIUS_final_cluster_assignment = function(CellSIUS.out, group_id, min_n_genes = 3){

  CellSIUS.out[,n_genes:=length(unique(gene_id)),by='sub_cluster']

  assignments = data.table::data.table(cell_idx = names(group_id), group=as.character(group_id))
  names(assignments) = c('cell_idx', 'group')
  assignments$group = as.character(assignments$group)
  assignments = merge(assignments, CellSIUS.out[n_genes>=min_n_genes,c('cell_idx','main_cluster','sub_cluster')],by='cell_idx',all=T)
  assignments = unique(assignments)

  final_assignment = function(main_cluster,sub_cluster){

    if(length(sub_cluster)==1){
      if(is.na(sub_cluster) || grepl("0$",sub_cluster)){
        out = main_cluster
      } else {
        out = gsub('_\\d$','',sub_cluster)
      }
    } else {
      subclusts = gsub('_\\d$', '',sub_cluster[grepl("1$",sub_cluster)])
      out = paste(subclusts,collapse='-')
      if(out == ''){out = main_cluster}
    }
    return(out)
  }

  assignments[,final:=final_assignment(group,sub_cluster),by="cell_idx"]
  assignments = unique(assignments[,c('cell_idx','final')])
  out = as.character(assignments$final)
  names(out)=assignments$cell_idx
  out = out[names(group_id)]
  return(out)
}

In [4]:
library(Matrix)
library(data.table)
data <- readMM("./ComicGTN/data/BMMC-bench-1/Gene_Cell.mtx")
Cell_names <- read.table("./ComicGTN/data/BMMC-bench-1/Cell_names.tsv", header = FALSE)
Gene_names <- read.table("./ComicGTN/data/BMMC-bench-1/Gene_names.tsv", header = FALSE)
data <- as.matrix(data)
rownames(data) <- Gene_names[,1]
colnames(data) <- Cell_names[,1]
clusters <- read.table("./ComicGTN/data/BMMC-bench-1/ini_clu.tsv", header = FALSE)
clusters <- clusters[,1]
clusters <- setNames(clusters,Cell_names[,1])
CellSIUS.out<-CellSIUS(mat.norm = data,group_id = clusters,min_n_cells=10, min_fc = 2, corr_cutoff = NULL, iter=0, max_perc_cells = 50, fc_between_cutoff = 1, mcl_path = "/usr/bin/mcl")
Result_List=CellSIUS_GetResults(CellSIUS.out=CellSIUS.out)
out=cellsius_get_cells_per_subclust(CellSIUS.out)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 3.4 GiB”


--------------------------------------------------------
 Summary of CellSIUS output
 --------------------------------------------------------

Main cluster:  0 
 ---------------
Subcluster:  1 
 Number of cells:  1 
 Marker genes: 
      gene_id   log2FC
       <char>    <num>
1:  ITFG2-AS1 2.821321
2: TMEM9B-AS1 2.744844


Main cluster:  1 
 ---------------
Subcluster:  1 
 Number of cells:  27 
 Marker genes: 
   gene_id   log2FC
    <char>    <num>
1:   ACAD8 2.798464
2:  TMEM97 2.613411


Main cluster:  10 
 ---------------
Subcluster:  1 
 Number of cells:  56 
 Marker genes: 
   gene_id   log2FC
    <char>    <num>
1:    AZU1 4.677326
2:   ELANE 3.470938
3:   PRTN3 3.030195


Subcluster:  2 
 Number of cells:  66 
 Marker genes: 
     gene_id   log2FC
      <char>    <num>
1:      GPC5 7.044419
2: IL12A-AS1 2.677148
3:     PREX2 4.465708


Main cluster:  2 
 ---------------
Subcluster:  1 
 Number of cells:  2 
 Marker genes: 
   gene_id   log2FC
    <char>    <num>
1:  IL17RC 2

In [6]:
a = CellSIUS_final_cluster_assignment(CellSIUS.out, clusters, min_n_genes = 3)
b <- prop.table(table(a)) * 100
low_prop_categories <- b[b < 1]
print(names(low_prop_categories))
dt <- data.table(name = names(a), value = a)
result <- dt$name[dt$value %in% names(low_prop_categories)]
print(result)
write.table(result, "./ComicGTN/data/BMMC-bench-1/CellSIUS_output.tsv",sep = "\t",row.names = FALSE, col.names = FALSE)

  [1] "10_1-10_11-10_28-10_29-10_27-10_33-10_8"                                                                
  [2] "10_1-10_20-10_36"                                                                                       
  [3] "10_1-10_21-10_12-10_8-10_20-10_30-10_18"                                                                
  [4] "10_1-10_21-10_17-10_27-10_9-10_20-10_26"                                                                
  [5] "10_1-10_21-10_24"                                                                                       
  [6] "10_1-10_21-10_28-10_29-10_8-10_18"                                                                      
  [7] "10_1-10_21-10_28-10_6-10_10-10_23-10_36"                                                                
  [8] "10_1-10_21-10_35-10_24-10_9-10_20-10_25-10_26-10_15"                                                    
  [9] "10_1-10_21-10_35-10_28-10_29-10_4-10_24-10_33-10_16-10_9-10_23-10_25-10_36"                      

  [1] "8"    "39"   "40"   "41"   "56"   "61"   "77"   "105"  "127"  "135" 
 [11] "186"  "187"  "202"  "204"  "240"  "242"  "252"  "278"  "296"  "303" 
 [21] "322"  "345"  "350"  "387"  "436"  "446"  "447"  "457"  "487"  "488" 
 [31] "502"  "520"  "523"  "554"  "563"  "622"  "625"  "627"  "674"  "678" 
 [41] "707"  "711"  "712"  "715"  "726"  "800"  "805"  "830"  "834"  "852" 
 [51] "864"  "892"  "919"  "931"  "942"  "966"  "993"  "1002" "1007" "1010"
 [61] "1028" "1038" "1072" "1154" "1184" "1188" "1191" "1207" "1219" "1261"
 [71] "1262" "1327" "1339" "1399" "1403" "1414" "1418" "1449" "1461" "1469"
 [81] "1534" "1573" "1585" "1592" "1620" "1651" "1672" "1674" "1676" "1724"
 [91] "1764" "1781" "1790" "1814" "1833" "1840" "1841" "1881" "1890" "1896"
[101] "1905" "1917" "1936" "1977" "1990" "1999" "2027" "2039" "2042" "2079"
[111] "2125" "2161" "2170" "2182" "2232" "2266" "2356" "2398" "2476" "2541"
[121] "2565" "2618" "2625" "2627" "2636" "2671" "2677" "2682" "2691" "2693"
[131] "2697"